In [1]:
import numpy as np
import pandas as pd
from f1winnerprediction import (
	config, 
	io_fastf1,
	mapper
)
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report,roc_auc_score, roc_curve, auc, mean_absolute_error, r2_score, accuracy_score, f1_score
import fastf1
import fastf1.core
import xgboost as xgb
from pprint import pprint

pd.set_option('display.max_columns', None)

fastf1.Cache.enable_cache(config.FASTF1_RAW_CACHE_DIR.as_posix())

# Load sessions from DB

In [2]:
# sessions: dict[int, list[fastf1.core.Session]] = io_fastf1.fetch_race_sessions_cache(use_sessions_cache=False, use_checkpoint=False)
# sessions

# Load sessions from dump files

In [3]:
sessions: dict[int, list[fastf1.core.Session]] = io_fastf1.load_sessions_from_years()
sessions

{2021: [2021 Season Round 1: Bahrain Grand Prix - Race,
  2021 Season Round 2: Emilia Romagna Grand Prix - Race,
  2021 Season Round 3: Portuguese Grand Prix - Race,
  2021 Season Round 4: Spanish Grand Prix - Race,
  2021 Season Round 5: Monaco Grand Prix - Race,
  2021 Season Round 6: Azerbaijan Grand Prix - Race,
  2021 Season Round 7: French Grand Prix - Race,
  2021 Season Round 8: Styrian Grand Prix - Race,
  2021 Season Round 9: Austrian Grand Prix - Race,
  2021 Season Round 10: British Grand Prix - Race,
  2021 Season Round 11: Hungarian Grand Prix - Race,
  2021 Season Round 12: Belgian Grand Prix - Race,
  2021 Season Round 13: Dutch Grand Prix - Race,
  2021 Season Round 14: Italian Grand Prix - Race,
  2021 Season Round 15: Russian Grand Prix - Race,
  2021 Season Round 16: Turkish Grand Prix - Race,
  2021 Season Round 17: United States Grand Prix - Race,
  2021 Season Round 18: Mexico City Grand Prix - Race,
  2021 Season Round 19: São Paulo Grand Prix - Race,
  2021 Sea

In [4]:
driver_mapping = io_fastf1.build_drivers_dict(sessions)
driver_mapping

{'VET': {'index': 21},
 'GAS': {'index': 19},
 'BOT': {'index': 23},
 'RUS': {'index': 19},
 'STR': {'index': 19},
 'GIO': {'index': 21},
 'ALO': {'index': 19},
 'RIC': {'index': 17},
 'NOR': {'index': 19},
 'VER': {'index': 19},
 'MSC': {'index': 21},
 'HAM': {'index': 19},
 'MAZ': {'index': 21},
 'LAT': {'index': 21},
 'LEC': {'index': 19},
 'PER': {'index': 23},
 'RAI': {'index': 21},
 'SAI': {'index': 19},
 'TSU': {'index': 19},
 'OCO': {'index': 19},
 'KUB': {'index': 13},
 'ZHO': {'index': 23},
 'HUL': {'index': 19},
 'ALB': {'index': 19},
 'MAG': {'index': 23},
 'DEV': {'index': 9},
 'SAR': {'index': 14},
 'PIA': {'index': 19},
 'LAW': {'index': 19},
 'BEA': {'index': 19},
 'COL': {'index': 19},
 'DOO': {'index': 5},
 'ANT': {'index': 19},
 'HAD': {'index': 19},
 'BOR': {'index': 19}}

In [5]:
import importlib

importlib.reload(mapper)


<module 'f1winnerprediction.mapper' from '/media/dhiabenhamouda/Dhia/Work/F1WinnerPrediction/src/f1winnerprediction/mapper.py'>

In [6]:
gp_index_map: dict[str, dict[int, int]] = {}
gp_index_map = mapper.map_gp_indices_between_years(sessions)
pprint(gp_index_map)

{'2021-2022': {0: 0,
               1: 3,
               3: 5,
               4: 6,
               5: 7,
               6: 11,
               8: 10,
               9: 9,
               10: 12,
               11: 13,
               12: 14,
               13: 15,
               16: 18,
               17: 19,
               18: 20,
               20: 1,
               21: 21},
 '2022-2023': {0: 0,
               1: 1,
               2: 2,
               4: 4,
               5: 6,
               6: 5,
               7: 3,
               8: 7,
               9: 9,
               10: 8,
               12: 10,
               13: 11,
               14: 12,
               15: 13,
               16: 14,
               17: 15,
               18: 17,
               19: 18,
               20: 19,
               21: 21},
 '2023-2024': {0: 0,
               1: 1,
               2: 2,
               3: 16,
               4: 5,
               5: 7,
               6: 9,
               7: 8,
            

# Prepare laptime data

In [7]:
years = config.YEARS_TO_FETCH
# Parse year by year
dataset = pd.DataFrame()
for year_first in tqdm(years):
	if (year_first+1) not in years:
		continue
	second_year = year_first + 1
 
	# Parse session by session
	for session_index, first_session in enumerate(sessions[year_first]):
    
		# Check if the GP index mapping exists
		gp_index_map_key = str(year_first) + "-" + str(second_year)
  
		if gp_index_map_key not in gp_index_map:
			print(f"GP: {gp_index_map_key} not found in mapping.")
			continue

		# Check if the session_index exists in the mapping
		# = if the GP was held in both years
		if session_index not in gp_index_map[gp_index_map_key]:
			print(f"Year: {year_first}, GP: {gp_index_map_key} has no mapped index.")
			continue
		
		try:
     
			# Get the mapped index
			mapped_index = gp_index_map[gp_index_map_key][session_index]
			print(f"Year: {year_first}, GP: {gp_index_map_key}, Mapped Index: {mapped_index}")
			second_session = sessions[second_year][mapped_index]
				
			# Now we have the current session and the corresponding session for next year
			mean_lap_time_first_year = first_session.laps["LapTime"].dt.total_seconds().groupby(first_session.laps["Driver"]).mean().sort_values()
			df_mean_lap_time_first_year = mean_lap_time_first_year.reset_index()
	
			mean_lap_time_latter_year = second_session.laps["LapTime"].dt.total_seconds().groupby(second_session.laps["Driver"]).mean().sort_values()
			df_mean_lap_time_next_year = mean_lap_time_latter_year.reset_index()	

			# Merge both DataFrames on 'Driver'
			merged_df = pd.merge(df_mean_lap_time_first_year, df_mean_lap_time_next_year, on="Driver", suffixes=('_first_year', '_latter_year'))
			print(merged_df)
		except Exception as e:
			print(f"An error occurred for Year: {year_first}, GP: {gp_index_map_key}, Mapped Index: {mapped_index}. Error: {e}")
			continue
		
		dataset = pd.concat([dataset, merged_df], ignore_index=True)
  
dataset.sort_values(by=['Driver'], inplace=True)
dataset

  0%|          | 0/5 [00:00<?, ?it/s]

Year: 2021, GP: 2021-2022, Mapped Index: 0
   Driver  LapTime_first_year  LapTime_latter_year
0     VER           97.575291           102.753434
1     HAM           97.586200           102.864193
2     BOT           98.310564           102.977246
3     NOR           98.514473           103.682789
4     LEC           98.687255           100.697709
5     RIC           98.888291           103.658930
6     PER           99.570429           102.955804
7     SAI           99.839232           102.792667
8     TSU          100.171232           103.052105
9     GAS          100.181490           100.781023
10    STR          100.189464           103.499246
11    OCO          100.727182           103.035211
12    RUS          100.760982           102.040107
13    MSC          101.830982           103.265930
14    ALO          102.290452           103.087263
15    LAT          102.558608           103.778579
Year: 2021, GP: 2021-2022, Mapped Index: 3
   Driver  LapTime_first_year  LapTime_latter_y

 20%|██        | 1/5 [00:00<00:01,  3.72it/s]

   Driver  LapTime_first_year  LapTime_latter_year
0     HAM           87.689400            91.186623
1     LEC           87.710980            91.131264
2     RIC           87.777429            88.488533
3     VER           88.549920            91.085113
4     NOR           89.179500            91.202226
5     PER           90.147275            91.200038
6     BOT           90.383314            92.844769
7     SAI           90.487294            91.180604
8     STR           90.619451            89.491103
9     ALO           90.677686            88.784484
10    RUS           90.682824            89.938673
11    OCO           90.690157            91.242151
12    LAT           90.770059            92.890442
13    VET           92.046923            88.822500
14    MSC           92.557808            91.246415
15    GAS          106.783000            91.205792
16    TSU                 NaN            92.862904
Year: 2021, GP: 2021-2022 has no mapped index.
Year: 2021, GP: 2021-2022 has no ma

 40%|████      | 2/5 [00:00<00:00,  4.55it/s]

...

 60%|██████    | 3/5 [00:00<00:00,  4.68it/s]

   Driver  LapTime_first_year  LapTime_latter_year
0     LEC           99.869689            98.805040
1     VER           99.957867            98.851020
2     RUS          100.396489            98.519380
3     OCO          100.759933           100.572020
4     GAS          100.989356           102.464867
5     PIA          101.140956            99.546680
6     ALB          101.280778           101.280840
7     PER          101.395630            99.781660
8     STR          101.584500           100.201420
9     MAG          101.701778            99.915440
10    ZHO          101.774356           100.001080
11    SAI          101.916957            98.757500
12    HAM          102.537370            98.665640
13    ALO          102.586609            99.903280
14    HUL          103.429488            99.715540
15    TSU          105.027977            99.775540
16    BOT          107.447020           100.760286
17    NOR          124.844500            99.387080
Year: 2023, GP: 2023-2024, Mapp

100%|██████████| 5/5 [00:00<00:00,  5.83it/s]


An error occurred for Year: 2024, GP: 2024-2025, Mapped Index: 22. Error: The data you are trying to access has not been loaded yet. See `Session.load`
Year: 2024, GP: 2024-2025, Mapped Index: 23
An error occurred for Year: 2024, GP: 2024-2025, Mapped Index: 23. Error: The data you are trying to access has not been loaded yet. See `Session.load`


,Driver,LapTime_first_year,LapTime_latter_year
829,ALB,114.696227,110.470250
620,ALB,100.465088,98.511214
1282,ALB,103.177630,102.145732
493,ALB,81.016153,84.988514
317,ALB,102.143717,100.944320
...,...,...,...
442,ZHO,72.793014,73.689143
378,ZHO,92.128581,85.764052
901,ZHO,101.666184,115.406273
641,ZHO,99.019080,98.048479


### Save new dataset

In [8]:
dataset.to_csv(config.LAP_TIMES_DATASET_CSV_PATH, index=False)

### Cleaning

In [9]:
clean_dataset = dataset.dropna()
clean_dataset

,Driver,LapTime_first_year,LapTime_latter_year
829,ALB,114.696227,110.470250
620,ALB,100.465088,98.511214
1282,ALB,103.177630,102.145732
493,ALB,81.016153,84.988514
317,ALB,102.143717,100.944320
...,...,...,...
442,ZHO,72.793014,73.689143
378,ZHO,92.128581,85.764052
901,ZHO,101.666184,115.406273
641,ZHO,99.019080,98.048479


### Training

In [10]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

def data_label_split(df_windows: pd.DataFrame):
	X = df_windows.iloc[:, :-1]
	y = df_windows.iloc[:, -1]
	return X, y

def normalize_features(X: pd.DataFrame):
   scaler = StandardScaler()
   X_normalized = scaler.fit_transform(X)
   return X_normalized


In [11]:
data = clean_dataset[["LapTime_first_year", "LapTime_latter_year"]]
X, y = data_label_split(data)
X = normalize_features(X)
X.shape, y.shape

((1244, 1), (1244,))

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
X_train.shape

(995, 1)

In [13]:
params = {'colsample_bytree': 1.0, 
          'learning_rate': 0.05,
          'max_depth': 8, 
          'n_estimators': 250, 
          'subsample': 1.0}
model = xgb.XGBRegressor(**params)
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,1.0
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [14]:
y_pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

MAE: 4.3374817459404404
R² Score: 0.616036903051147


In [15]:
session_last_year = sessions[2023][0]  # First session of 2023
pprint(session_last_year.laps.columns)

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')


In [16]:
mean_lap_time_first_year = session_last_year.laps["LapTime"].dt.total_seconds().groupby(session_last_year.laps["Driver"]).mean().sort_values()
df_mean_lap_time_next_year = mean_lap_time_first_year.reset_index()
df_mean_lap_time_next_year

,Driver,LapTime
0,VER,98.890105
1,PER,99.100404
2,ALO,99.567947
3,LEC,99.644051
4,SAI,99.733123
5,HAM,99.784439
6,STR,99.846281
7,RUS,99.870333
8,BOT,100.164614
9,GAS,100.184018


In [17]:
session_current_year = sessions[2024][0]  # First session of 2024

In [18]:
mean_lap_time_current_year = session_current_year.laps["LapTime"].dt.total_seconds().groupby(session_current_year.laps["Driver"]).mean().sort_values()
df_lap_time_current_year = mean_lap_time_current_year.reset_index()
df_lap_time_current_year

,Driver,LapTime
0,VER,96.574421
1,PER,96.968404
2,SAI,97.014947
3,LEC,97.270368
4,RUS,97.395263
5,NOR,97.424561
6,HAM,97.457298
7,PIA,97.558316
8,ALO,97.888228
9,STR,98.209789


In [19]:
data = df_mean_lap_time_next_year.merge(df_lap_time_current_year, on="Driver", suffixes=('_2023', '_2024'))
data


,Driver,LapTime_2023,LapTime_2024
0,VER,98.890105,96.574421
1,PER,99.100404,96.968404
2,ALO,99.567947,97.888228
3,LEC,99.644051,97.270368
4,SAI,99.733123,97.014947
5,HAM,99.784439,97.457298
6,STR,99.846281,98.209789
7,RUS,99.870333,97.395263
8,BOT,100.164614,98.503745
9,GAS,100.184018,98.877839


In [20]:
def get_laps(session: fastf1.core.Session) -> pd.DataFrame:
	"""
	Extracts lap time data for all drivers from a given session.

	Args:
		session: A fastf1 session object.

	Returns:
		A pandas DataFrame with Driver, LapNumber, and LapTime in seconds.
	"""
	if not session.laps.empty:
		laps = session.laps.copy()
		laps["LapTime"] = laps["LapTime"].dt.total_seconds()
		return laps[["Driver", "LapNumber", "LapTime"]].dropna()
	return pd.DataFrame()

def get_comparison_data(current_session: fastf1.core.Session, last_year_session: fastf1.core.Session) -> pd.DataFrame:
	"""
	Merges lap times from the same event in two consecutive years.

	Args:
		current_session: The session object for the current year.
		last_year_session: The session object for the previous year.

	Returns:
		A merged DataFrame with lap times from both years aligned by driver and lap number.
	"""
	laps_current = get_laps(current_session)
	laps_last_year = get_laps(last_year_session)

	if laps_current.empty or laps_last_year.empty:
		return pd.DataFrame()

	# Merge data on Driver and LapNumber
	merged_laps = pd.merge(
		laps_last_year, 
		laps_current, 
		on=["Driver", "LapNumber"], 
		suffixes=('_last_year', '_current_year')
	)
	return merged_laps

# --- Main processing loop ---
all_laps_data = []
# Assuming config.YEARS_TO_FETCH is sorted, start from the second year
for year in tqdm(sorted(config.YEARS_TO_FETCH)[1:-1], desc="Processing Years"):
	last_year = year - 1
	
	# Create a quick lookup map for last year's sessions by event name
	last_year_session_map = {
		s.event.EventName: s for s in sessions.get(last_year, [])
	}

	for current_session in sessions.get(year, []):
		# Find the corresponding session from the previous year
		last_year_session = last_year_session_map.get(current_session.event.EventName)
		print(f"Current Year: {year}, Event: {current_session.event.EventName}")
		if last_year_session:
			comparison_df = get_comparison_data(current_session, last_year_session)
			if not comparison_df.empty:
				all_laps_data.append(comparison_df)

# Concatenate all data into a single DataFrame
if all_laps_data:
	training_data = pd.concat(all_laps_data, ignore_index=True)
	print(f"Successfully created training data with {len(training_data)} samples.")
	print(training_data.head())
else:
	print("No matching sessions found to create training data.")
	training_data = pd.DataFrame()

Processing Years:   0%|          | 0/3 [00:00<?, ?it/s]

Current Year: 2022, Event: Bahrain Grand Prix
Current Year: 2022, Event: Saudi Arabian Grand Prix
Current Year: 2022, Event: Australian Grand Prix
Current Year: 2022, Event: Emilia Romagna Grand Prix
Current Year: 2022, Event: Miami Grand Prix
Current Year: 2022, Event: Spanish Grand Prix
Current Year: 2022, Event: Monaco Grand Prix
Current Year: 2022, Event: Azerbaijan Grand Prix
Current Year: 2022, Event: Canadian Grand Prix
Current Year: 2022, Event: British Grand Prix
Current Year: 2022, Event: Austrian Grand Prix
Current Year: 2022, Event: French Grand Prix
Current Year: 2022, Event: Hungarian Grand Prix
Current Year: 2022, Event: Belgian Grand Prix
Current Year: 2022, Event: Dutch Grand Prix
Current Year: 2022, Event: Italian Grand Prix
Current Year: 2022, Event: Singapore Grand Prix
Current Year: 2022, Event: Japanese Grand Prix
Current Year: 2022, Event: United States Grand Prix
Current Year: 2022, Event: Mexico City Grand Prix
Current Year: 2022, Event: São Paulo Grand Prix


Processing Years:  33%|███▎      | 1/3 [00:00<00:00,  4.37it/s]

Current Year: 2022, Event: Abu Dhabi Grand Prix
Current Year: 2023, Event: Bahrain Grand Prix
Current Year: 2023, Event: Saudi Arabian Grand Prix
Current Year: 2023, Event: Australian Grand Prix
Current Year: 2023, Event: Azerbaijan Grand Prix
Current Year: 2023, Event: Miami Grand Prix
Current Year: 2023, Event: Monaco Grand Prix
Current Year: 2023, Event: Spanish Grand Prix
Current Year: 2023, Event: Canadian Grand Prix
Current Year: 2023, Event: Austrian Grand Prix
Current Year: 2023, Event: British Grand Prix
Current Year: 2023, Event: Hungarian Grand Prix
Current Year: 2023, Event: Belgian Grand Prix
Current Year: 2023, Event: Dutch Grand Prix
Current Year: 2023, Event: Italian Grand Prix
Current Year: 2023, Event: Singapore Grand Prix
Current Year: 2023, Event: Japanese Grand Prix
Current Year: 2023, Event: Qatar Grand Prix
Current Year: 2023, Event: United States Grand Prix
Current Year: 2023, Event: Mexico City Grand Prix
Current Year: 2023, Event: São Paulo Grand Prix
Current 

Processing Years:  67%|██████▋   | 2/3 [00:00<00:00,  3.66it/s]

Current Year: 2024, Event: Bahrain Grand Prix
Current Year: 2024, Event: Saudi Arabian Grand Prix
Current Year: 2024, Event: Australian Grand Prix
Current Year: 2024, Event: Japanese Grand Prix
Current Year: 2024, Event: Chinese Grand Prix
Current Year: 2024, Event: Miami Grand Prix
Current Year: 2024, Event: Emilia Romagna Grand Prix
Current Year: 2024, Event: Monaco Grand Prix
Current Year: 2024, Event: Canadian Grand Prix
Current Year: 2024, Event: Spanish Grand Prix
Current Year: 2024, Event: Austrian Grand Prix
Current Year: 2024, Event: British Grand Prix
Current Year: 2024, Event: Hungarian Grand Prix
Current Year: 2024, Event: Belgian Grand Prix
Current Year: 2024, Event: Dutch Grand Prix
Current Year: 2024, Event: Italian Grand Prix
Current Year: 2024, Event: Azerbaijan Grand Prix
Current Year: 2024, Event: Singapore Grand Prix


Processing Years: 100%|██████████| 3/3 [00:00<00:00,  3.66it/s]

Current Year: 2024, Event: United States Grand Prix
Current Year: 2024, Event: Mexico City Grand Prix
Current Year: 2024, Event: São Paulo Grand Prix
Current Year: 2024, Event: Las Vegas Grand Prix
Current Year: 2024, Event: Qatar Grand Prix
Current Year: 2024, Event: Abu Dhabi Grand Prix
Successfully created training data with 49750 samples.
  Driver  LapNumber  LapTime_last_year  LapTime_current_year
0    HAM        1.0            119.538               101.555
1    HAM        2.0            142.712                99.002
2    HAM        4.0            104.932                98.892
3    HAM        5.0            105.139                98.923
4    HAM        6.0             96.169                99.707


In [21]:
training_data

,Driver,LapNumber,LapTime_last_year,LapTime_current_year
0,HAM,1.0,119.538,101.555
1,HAM,2.0,142.712,99.002
2,HAM,4.0,104.932,98.892
3,HAM,5.0,105.139,98.923
4,HAM,6.0,96.169,99.707
...,...,...,...,...
49745,MAG,53.0,90.445,90.215
49746,MAG,54.0,90.207,91.204
49747,MAG,55.0,90.252,91.958
49748,MAG,56.0,90.429,112.994


In [22]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

def data_label_split(df_windows: pd.DataFrame):
	X = df_windows.iloc[:, :-1]
	y = df_windows.iloc[:, -1]
	return X, y

def normalize_features(X: pd.DataFrame):
   scaler = StandardScaler()
   X_normalized = scaler.fit_transform(X)
   return X_normalized


In [32]:
data = training_data[["LapTime_last_year", "LapTime_current_year"]]
X, y = data_label_split(data)
X = normalize_features(X)
X.shape, y.shape

((49750, 1), (49750,))

In [53]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
X_train.shape

(39800, 1)

In [54]:
params = {'colsample_bytree': 1.0, 
          'learning_rate': 0.05,
          'max_depth': 8, 
          'n_estimators': 250, 
          'subsample': 1.0}
model = xgb.XGBRegressor(**params)
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,1.0
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [55]:
X_test.mean()

np.float64(-0.011291671494322263)

In [56]:
y_pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

MAE: 6.9778566216458255
R² Score: 0.04583937880311373


In [57]:
y_pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

MAE: 6.9778566216458255
R² Score: 0.04583937880311373


In [58]:
pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})

,Actual,Predicted
27771,75.390,83.590958
15528,82.720,97.514534
3578,108.949,83.694176
33501,76.561,99.293877
36141,81.139,82.857925
...,...,...
36416,78.385,85.787079
12111,78.000,83.618111
31508,83.070,100.138412
43873,108.467,105.612625
